# Bank Complaint Resolution Picker — Data Analysis & Recommendation Engine
This notebook loads CFPB Consumer Complaint Database records for four major
banks, computes how often each bank resolves complaints with relief by issue
type, and builds a quiz-based recommendation engine that ranks banks based
on a consumer's stated priorities.

## 1. Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# Constructing the file path to the CSV in Google Drive
file_path = '/content/drive/MyDrive/complaints-2026-09-26_14_53.csv'

# Reading the CSV into a pandas DataFrame
df = pd.read_csv(file_path)

# Displaying the first 5 rows of the DataFrame
display(df.head())

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:29:55.000Z,Credit card or prepaid card,Store credit card,"Other features, terms, or problems",Privacy issues,NaN,CAPITAL ONE FINANCIAL CORPORATION,FL,33617,NaN,Web,2023-03-11T14:58:19.000Z,Closed with non-monetary relief,Yes,6681516
1,2023-03-11T14:52:52.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem when making payments,You never received your bill or did not know a...,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",CA,92584,NaN,Web,2023-03-11T15:04:31.000Z,Closed with explanation,No,6681480
2,2023-03-11T15:19:39.000Z,Credit card or prepaid card,General-purpose credit card or charge card,"Other features, terms, or problems",Add-on products and services,NaN,JPMORGAN CHASE & CO.,NY,10006,NaN,Web,2023-03-11T15:48:57.000Z,Closed with explanation,Yes,6681404
3,2023-03-11T15:59:56.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,NaN,JPMORGAN CHASE & CO.,CA,92110,NaN,Web,2023-03-11T16:59:33.000Z,Closed with explanation,Yes,6681185
4,2023-03-11T15:52:34.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Fees or interest,Charged too much interest,NaN,CAPITAL ONE FINANCIAL CORPORATION,NY,11373,NaN,Web,2023-05-01T11:32:08.000Z,Closed with explanation,Yes,6681317


## 2. Data Exploration
Confirming the dataset only contains our four target banks, and checking the exact
company name strings (CFPB naming is inconsistent, e.g. commas/legal suffixes)
before filtering.

In [ ]:
# Making sure that there are only 4 unique companies in the dataset
print("Unique Companies:\n", df['Company'].unique())

Unique Companies:
 ['CAPITAL ONE FINANCIAL CORPORATION'
 'BANK OF AMERICA, NATIONAL ASSOCIATION' 'JPMORGAN CHASE & CO.'
 'WELLS FARGO & COMPANY']


## 3. Company Response Rates by Issue (Exploratory)
Computing, for each bank and each specific `Issue` category, the percentage of
complaints closed with each response type. This is an early exploration step. See note below about how this table relates to the final analysis.

In [ ]:

# Defining a list of companies that are exactly formatted as they appear in
# the dataset.
banks = ['JPMORGAN CHASE & CO.', 'BANK OF AMERICA, NATIONAL ASSOCIATION',
         'WELLS FARGO & COMPANY', 'CAPITAL ONE FINANCIAL CORPORATION']

# Checking if each company in the dataset is in the defined list of 4 target
# banks.
df_4banks = df[df['Company'].isin(banks)].copy()

# Spliting data into buckets of unqiue company and issue pairs and calculating
# how often each response type occurs
response_pct = (
    df_4banks
    .groupby(['Company', 'Issue'])['Company response to consumer']
    .value_counts(normalize=True) # converting to percentages
    .unstack(fill_value=0)
)
print(response_pct)

# NOTE: This table (response_pct) is built BEFORE we filter out "In progress" complaints
# (that filter happens later, in the cell below). That means the percentages here include
# "In progress" complaints in the denominator, which slightly understates the true
# resolution rates. This table is only used as an early sanity-check / exploration step,
# not in the final recommendation logic below.

Company response to consumer                                                              Closed with explanation  \
Company                               Issue                                                                         
BANK OF AMERICA, NATIONAL ASSOCIATION Advertising and marketing, including promotiona...                 0.651515   
                                      Closing your account                                               0.802534   
                                      Credit monitoring or identity theft protection ...                 0.714286   
                                      Fees or interest                                                   0.474754   
                                      Getting a credit card                                              0.566762   
...                                                                                                           ...   
WELLS FARGO & COMPANY                 Problem with a purchase sh

### Adding complaint counts and filtering out low-volume issue/company pairs

In [ ]:
# Counting how many total rows there are in each unqiue company + issue bucket and adding this as a new column onto percentage table
counts = df_4banks.groupby(['Company', 'Issue']).size().rename('n_complaints')
response_pct = response_pct.join(counts)
response_pct = response_pct[response_pct['n_complaints'] >= 20]  # filtering out any buckets with less than 20 complaints because that's too small of a sample

### Removing unresolved ("In progress") complaints

In [ ]:
# Dropping unresolved complaints from the dataset
df_4banks = df_4banks[df_4banks['Company response to consumer'] != 'In progress']

# Dropping "In progress" complaints here because they have no final outcome yet -
# counting them would understate relief rates in any analysis done after this point.
# IMPORTANT: this filter is applied AFTER response_pct was already computed,
# so response_pct still includes "In progress" complaints and is NOT directly comparable
# to concern_stats (built below, after this filter). Only concern_stats reflects
# fully-resolved complaints and is what recommend_banks() uses.

### Sanity check: largest issue/company pairs by volume

In [ ]:
# Viewing few entries of the table for a sanity check
response_pct.sort_values('n_complaints', ascending=False).head(20)

Closed with explanation  \
Company                               Issue                                                                         
JPMORGAN CHASE & CO.                  Problem with a purchase shown on your statement                    0.741075   
CAPITAL ONE FINANCIAL CORPORATION     Problem with a purchase shown on your statement                    0.674463   
BANK OF AMERICA, NATIONAL ASSOCIATION Problem with a purchase shown on your statement                    0.460836   
CAPITAL ONE FINANCIAL CORPORATION     Getting a credit card                                              0.654729   
                                      Other features, terms, or problems                                 0.759016   
                                      Fees or interest                                                   0.636657   
JPMORGAN CHASE & CO.                  Other features, terms, or problems                                 0.701077   
                                      Getting a credit card                                              0.686118   
CAPITAL ONE FINANCIAL CORPORATION     Closing your account                                               0.855828   
WELLS FARGO & COMPANY                 Problem with a purchase shown on your statement                    0.707740   
CAPITAL ONE FINANCIAL CORPORATION     Problem when making payments                                       0.806598   
BANK OF AMERICA, NATIONAL ASSOCIATION Getting a credit card                                              0.566762   
CAPITAL ONE FINANCIAL CORPORATION     Incorrect information on your report                               0.806537   
BANK OF AMERICA, NATIONAL ASSOCIATION Other features, terms, or problems                                 0.601272   
CAPITAL ONE FINANCIAL CORPORATION     Trouble using your card                                            0.900415   
JPMORGAN CHASE & CO.                  Closing your account                                               0.876350   
BANK OF AMERICA, NATIONAL ASSOCIATION Problem when making payments                                       0.550651   
JPMORGAN CHASE & CO.                  Fees or interest                                                   0.721402   
                                      Advertising and marketing, including promotiona...                 0.768125   
BANK OF AMERICA, NATIONAL ASSOCIATION Fees or interest                                                   0.474754   

                                                                                          Closed with monetary relief  \
Company                               Issue                                                                             
JPMORGAN CHASE & CO.                  Problem with a purchase shown on your statement                        0.222299   
CAPITAL ONE FINANCIAL CORPORATION     Problem with a purchase shown on your statement                        0.281384   
BANK OF AMERICA, NATIONAL ASSOCIATION Problem with a purchase shown on your statement                        0.527322   
CAPITAL ONE FINANCIAL CORPORATION     Getting a credit card                                                  0.006244   
                                      Other features, terms, or problems                                     0.137330   
                                      Fees or interest                                                       0.326389   
JPMORGAN CHASE & CO.                  Other features, terms, or problems                                     0.151842   
                                      Getting a credit card                                                  0.007368   
CAPITAL ONE FINANCIAL CORPORATION     Closing your account                                                   0.054908   
WELLS FARGO & COMPANY                 Problem with a purchase shown on your statement                        0.193808   
CAPITAL ONE FINANCIAL CORPORATION     Problem when making payme

## 4. Grouping Issues into Consumer-Facing Concern Categories
Maping the ~20 raw `Issue` values into 7 broader, consumer-friendly categories
(e.g. "Billing/Disputes," "Customer Service"), then recomputing relief rates
at this coarser level. This table (`concern_stats`) is the one used by the
recommendation engine below (unlike the exploratory table in Section 3,
this one excludes "In progress" complaints).

In [ ]:
# Creating a dictionary to group related issues within the dataset to braoder, consumer-friendly categoriesconcern_map =
concern_map = {
    'Problem with a purchase shown on your statement': 'Billing/Disputes',
    'Fees or interest': 'Billing/Disputes',
    'Incorrect information on your report': 'Billing/Disputes',
    'Getting a credit card': 'Application/Approval',
    'Closing your account': 'Account Management',
    'Trouble using your card': 'Customer Service',
    'Problem when making payments': 'Payments',
    'Advertising and marketing, including promotional offers': 'Marketing',
}
df_4banks['concern'] = df_4banks['Issue'].map(concern_map)

# Issues not in concern_map become NaN here - we exclude them rather than force-fit
# them into one of the 6 named categories used by the website.
n_excluded = df_4banks['concern'].isna().sum()
print(f"Excluding {n_excluded} complaints that didn't map to a named concern category")
df_4banks = df_4banks[df_4banks['concern'].notna()]

# Regrouping the data by broader, consumer-friendly category
concern_stats = (
    df_4banks
    .groupby(['Company', 'concern'])
    .agg(
        n=('Complaint ID', 'count'),
        monetary_relief_rate=('Company response to consumer', lambda x: (x == 'Closed with monetary relief').mean()),
        non_monetary_relief_rate=('Company response to consumer', lambda x: (x == 'Closed with non-monetary relief').mean()),
    )
    .reset_index()
)

# Defining total relief rate to be a combination of both monetary and nonmonetary relief
concern_stats['total_relief_rate'] = concern_stats['monetary_relief_rate'] + concern_stats['non_monetary_relief_rate']
print(concern_stats.sort_values(['concern', 'total_relief_rate'], ascending=[True, False]))

Excluding 16615 complaints that didn't map to a named concern category
                                  Company               concern      n  \
18                  WELLS FARGO & COMPANY    Account Management    613   
0   BANK OF AMERICA, NATIONAL ASSOCIATION    Account Management   1342   
6       CAPITAL ONE FINANCIAL CORPORATION    Account Management   3260   
12                   JPMORGAN CHASE & CO.    Account Management   1852   
1   BANK OF AMERICA, NATIONAL ASSOCIATION  Application/Approval   2449   
7       CAPITAL ONE FINANCIAL CORPORATION  Application/Approval   5445   
19                  WELLS FARGO & COMPANY  Application/Approval   1437   
13                   JPMORGAN CHASE & CO.  Application/Approval   3393   
2   BANK OF AMERICA, NATIONAL ASSOCIATION      Billing/Disputes   8151   
20                  WELLS FARGO & COMPANY      Billing/Disputes   4587   
8       CAPITAL ONE FINANCIAL CORPORATION      Billing/Disputes  14676   
14                   JPMORGAN CHASE & CO.

## 5. Recommendation Function
Given a set of concerns a consumer cares about (and optional importance
weights), ranking the four banks by weighted average relief rate.

In [ ]:
# Filtering concern_stats table down to only the rows matching whatever concerns the consumer cares about
def recommend_banks(selected_concerns, stats_df, weights=None):
    """
    selected_concerns: list of concern strings the consumer picked, e.g. ['Billing/Disputes', 'Customer Service']
    weights: optional dict mapping concern -> importance weight; defaults to equal weight
    """
    subset = stats_df[stats_df['concern'].isin(selected_concerns)].copy()

    # Treating every concern as equally importatn if no weights are given
    if weights is None:
        weights = {c: 1 for c in selected_concerns}
    subset['weight'] = subset['concern'].map(weights)

    # Calculating the weighted average relief rate per company across selected concerns
    # Standard Weighted-Average formula = multiplies each concern's relief rate by its weight, sums those up, then divides by the total weight
    ranked = (
        subset.groupby('Company')
        .apply(lambda g: (g['total_relief_rate'] * g['weight']).sum() / g['weight'].sum())
        .sort_values(ascending=False)
        .rename('weighted_relief_score')
        .reset_index()
    )
    return ranked

## 6. Quiz Design
Designing four questions used to infer which concern categories matter most to a six given consumer, without asking them to pick categories directly.

In [ ]:
# Defining the consumer questions prompts and answers
quiz_questions = {
    'q1': {
        'prompt': "How important is it that the application process is clear, with no surprises?",
        'concern': 'Application/Approval',
        'options': ['Very important', 'Somewhat important', 'Not important']
    },
    'q2': {
        'prompt': "How much do rewards, cash back, or sign-up bonuses matter in choosing this card?",
        'concern': 'Marketing',
        'options': ['A lot', 'Somewhat', 'Not much']
    },
    'q3': {
        'prompt': "Have you dealt with card fraud or a billing dispute before?",
        'concern': 'Billing/Disputes',
        'options': ['Yes, and it was a hassle to resolve', 'Yes, but it went fine', 'No']
    },
    'q4': {
        'prompt': "Do you expect to carry a balance month-to-month?",
        'concern': 'Payments',
        'options': ['Yes, regularly', 'Sometimes', 'No, I pay in full']
    },
    'q5': {
        'prompt': "How important is being able to quickly reach a live person?",
        'concern': 'Customer Service',
        'options': ['Very important', 'Somewhat important', 'Not important']
    },
    'q6': {
        'prompt': "Do you think you'll need to make changes to the account itself later on?",
        'concern': 'Account Management',
        'options': ['Likely', 'Maybe', 'Unlikely']
    },
}

### Converting quiz answers into concern weights
Each answer maps to weight boosts for one or more concern categories. Where
multiple answers point to the same concern, we take the maximum boost rather
than stacking them additively.

In [ ]:
# Starting every concern at natural weight of 0
def compute_concern_weights(answers):
    """
    answers: dict like {'q1': 'Very important', 'q2': 'Somewhat', ...}
    Returns a weight dict over the 6 concern buckets. Each question maps
    to exactly one concern (see quiz_questions above), so no merging
    across questions is needed.
    """
    weights = {
        'Account Management': 1.0,
        'Application/Approval': 1.0,
        'Billing/Disputes': 1.0,
        'Customer Service': 1.0,
        'Marketing': 1.0,
        'Payments': 1.0,
    }

# For each quiz question answer, a dictionary maps the consumer answer to which concerns shoulld get boosted and by how much.
    answer_to_weight = {
        'Very important': 2.5, 'A lot': 2.5, 'Yes, regularly': 2.0, 'Likely': 2.5,
        'Yes, and it was a hassle to resolve': 2.5,
        'Somewhat important': 1.5, 'Somewhat': 1.5, 'Sometimes': 1.3, 'Maybe': 1.5,
        'Yes, but it went fine': 1.3,
        'Not important': 1.0, 'Not much': 1.0, 'No, I pay in full': 1.0, 'Unlikely': 1.0,
        'No': 1.0,
    }

    for q_id, answer in answers.items():
        concern = quiz_questions[q_id]['concern']
        val = answer_to_weight.get(answer, 1.0)
        weights[concern] = max(weights[concern], val)

    return weights

### Tying it together: quiz answers → weights → ranked banks

In [ ]:
# Defining the overall function to computes weights from the quiz answers, keeps only the concerns
# that actually got boosted above the neutral 1.0 (active_concerns — no point including concerns
# nobody indicated they care about), then calls your ranking function with those concerns and weights
def recommend_banks_from_quiz(answers, stats_df):
    weights = compute_concern_weights(answers)
    active_concerns = [c for c, w in weights.items() if w > 1.0]  # only concerns the quiz actually flagged
    return recommend_banks(active_concerns, stats_df, weights=weights)